In [ ]:
# Cell 1: Environment & Quant Library Imports
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import talib
import quantstats as qs
import vectorbt as vbt

print("Quant Discovery Environment Loaded Successfully.")

In [ ]:
# Cell 2: Data Ingestion (2-Year Synthetic 1-Hour BTC/USD OHLCV Dataset)
rng = np.random.default_rng(seed=42)
n_hours = 2 * 365 * 24  # 17,520 hourly candles (2 years)

timestamps = pd.date_range(
    start="2024-01-01 00:00:00",
    periods=n_hours,
    freq="1h",
    tz="UTC",
)

# Simulate hourly log returns with cyclical drift + volatility clustering + mean-reverting pullbacks
t = np.arange(n_hours)
regime_drift = 0.00006 + 0.00025 * np.sin(2 * np.pi * t / (24 * 30))
vol_cluster = 0.0065 * (1.0 + 0.45 * np.sin(2 * np.pi * t / (24 * 7)))
shocks = rng.normal(loc=regime_drift, scale=vol_cluster, size=n_hours)

close_prices = 42000.0 * np.exp(np.cumsum(shocks))
open_prices = np.roll(close_prices, 1)
open_prices[0] = 42000.0

intrabar_spread = close_prices * rng.uniform(0.0015, 0.0090, size=n_hours)
high_prices = np.maximum(open_prices, close_prices) + intrabar_spread
low_prices = np.minimum(open_prices, close_prices) - intrabar_spread
volumes = rng.lognormal(mean=4.5, sigma=0.65, size=n_hours)

df = pd.DataFrame(
    {
        "open": open_prices,
        "high": high_prices,
        "low": low_prices,
        "close": close_prices,
        "volume": volumes,
    },
    index=timestamps,
)

print(f"Generated {len(df):,} hourly BTC/USD candles from {df.index[0]} to {df.index[-1]}")
df.tail()

In [ ]:
# Cell 3: Baseline Strategy (Iteration 1: 14-Period RSI < 40 AND MACD > MACD_Signal)
close_arr = df["close"].to_numpy(dtype=np.float64)

# Calculate indicators strictly via TA-Lib
df["rsi_14"] = talib.RSI(close_arr, timeperiod=14)
macd, macd_signal, macd_hist = talib.MACD(
    close_arr,
    fastperiod=12,
    slowperiod=26,
    signalperiod=9,
)
df["macd"] = macd
df["macd_signal"] = macd_signal
df["macd_hist"] = macd_hist

# Long-only entry signal: RSI < 40 AND MACD > MACD_Signal
df["entry_signal"] = (df["rsi_14"] < 40.0) & (df["macd"] > df["macd_signal"])

# Exit when momentum reverses (MACD crosses below signal or RSI > 65)
df["exit_signal"] = (df["macd"] < df["macd_signal"]) | (df["rsi_14"] > 65.0)

# Stateful long-only position tracking (1.0 = in position, 0.0 = flat)
position = np.zeros(len(df), dtype=np.float64)
entries = df["entry_signal"].to_numpy()
exits = df["exit_signal"].to_numpy()
in_trade = 0.0
for i in range(len(df)):
    if in_trade == 0.0 and entries[i]:
        in_trade = 1.0
    elif in_trade == 1.0 and exits[i]:
        in_trade = 0.0
    position[i] = in_trade

df["position"] = position

# Shift position by 1 bar to prevent lookahead bias when calculating hourly strategy returns
df["asset_returns"] = df["close"].pct_change().fillna(0.0)
df["strategy_returns"] = df["position"].shift(1).fillna(0.0) * df["asset_returns"]

# Strip timezone on series index for QuantStats compatibility
strategy_returns = df["strategy_returns"].copy()
strategy_returns.index = strategy_returns.index.tz_localize(None)

total_signals = int(df["entry_signal"].sum())
bars_in_market = int(df["position"].sum())
print(f"Entry Signals Triggered: {total_signals} | Hourly Bars in Market: {bars_in_market}")

In [ ]:
# Cell 4: Strategy Evaluation via QuantStats (Annualized for 24/7 Hourly Crypto = 8,760 periods/year)
HOURS_PER_YEAR = 365 * 24

sharpe_ratio = float(qs.stats.sharpe(strategy_returns, periods=HOURS_PER_YEAR))
win_rate = float(qs.stats.win_rate(strategy_returns))
cumulative_return = float((1.0 + strategy_returns).prod() - 1.0)

print("=== Baseline Strategy Performance (Iteration 1) ===")
print(f"Sharpe Ratio (Annualized): {sharpe_ratio:.4f}")
print(f"Win Rate (Active Bars):    {win_rate * 100:.2f}%")
print(f"Cumulative Return:         {cumulative_return * 100:.2f}%")

In [ ]:
# Cell 5: Iteration 2 Strategy (EMA_200 Trend Filter + Multi-Horizon MACD + StochRSI + ROC)
close_arr = df["close"].to_numpy(dtype=np.float64)

# 1. Macro Trend Filter: 200-period Exponential Moving Average
df["ema_200"] = talib.EMA(close_arr, timeperiod=200)

# 2. Optimized Momentum Indicators (TA-Lib)
# Slower MACD (48, 104, 36) captures multi-day swing waves rather than 1-hour noise
macd_2, macd_signal_2, macd_hist_2 = talib.MACD(
    close_arr,
    fastperiod=48,
    slowperiod=104,
    signalperiod=36,
)
df["macd_2"] = macd_2
df["macd_signal_2"] = macd_signal_2

# Stochastic RSI (14, 5, 3) for precise pullback timing inside confirmed uptrends
stoch_k, stoch_d = talib.STOCHRSI(
    close_arr,
    timeperiod=14,
    fastk_period=5,
    fastd_period=3,
    fastd_matype=0,
)
df["stoch_k"] = stoch_k
df["stoch_d"] = stoch_d

# 72-hour Rate of Change confirms positive 3-day drift
df["roc_72"] = talib.ROC(close_arr, timeperiod=72)

# Long-only entry: Price > EMA_200 + bullish multi-day MACD + positive 72h ROC + StochRSI K > D
df["entry_signal_2"] = (
    (df["close"] > df["ema_200"])
    & (df["macd_2"] > df["macd_signal_2"])
    & (df["macd_2"] > 0.0)
    & (df["roc_72"] > 0.0)
    & (df["stoch_k"] > df["stoch_d"])
)

# Exit when price breaks below EMA_200 or multi-day MACD turns negative
df["exit_signal_2"] = (df["close"] < df["ema_200"]) | (df["macd_2"] < 0.0)

# Stateful long-only position tracking
position_2 = np.zeros(len(df), dtype=np.float64)
entries_2 = df["entry_signal_2"].to_numpy()
exits_2 = df["exit_signal_2"].to_numpy()
in_trade_2 = 0.0
for i in range(250, len(df)):
    if in_trade_2 == 0.0 and entries_2[i]:
        in_trade_2 = 1.0
    elif in_trade_2 == 1.0 and exits_2[i]:
        in_trade_2 = 0.0
    position_2[i] = in_trade_2

df["position_2"] = position_2

# 1-bar shift to prevent lookahead bias
df["strategy_returns_2"] = df["position_2"].shift(1).fillna(0.0) * df["asset_returns"]

strategy_returns_2 = df["strategy_returns_2"].copy()
strategy_returns_2.index = strategy_returns_2.index.tz_localize(None)

total_signals_2 = int(df["entry_signal_2"].sum())
bars_in_market_2 = int(df["position_2"].sum())
print(f"Iteration 2 Entry Signals: {total_signals_2} | Hourly Bars in Market: {bars_in_market_2}")

In [ ]:
# Cell 6: Iteration 2 Evaluation via QuantStats
HOURS_PER_YEAR = 365 * 24

sharpe_ratio_2 = float(qs.stats.sharpe(strategy_returns_2, periods=HOURS_PER_YEAR))
win_rate_2 = float(qs.stats.win_rate(strategy_returns_2))
cumulative_return_2 = float((1.0 + strategy_returns_2).prod() - 1.0)

print("=== Evolved Strategy Performance (Iteration 2) ===")
print(f"Sharpe Ratio (Annualized): {sharpe_ratio_2:.4f}")
print(f"Win Rate (Active Bars):    {win_rate_2 * 100:.2f}%")
print(f"Cumulative Return:         {cumulative_return_2 * 100:.2f}%")